In [1]:
!pip install --no-index --find-links=/kaggle/input/ariel-2024-pqdm pqdm

Looking in links: /kaggle/input/ariel-2024-pqdm
Processing /kaggle/input/ariel-2024-pqdm/pqdm-0.2.0-py2.py3-none-any.whl
Processing /kaggle/input/ariel-2024-pqdm/bounded_pool_executor-0.0.3-py3-none-any.whl (from pqdm)


In [2]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from pqdm.threads import pqdm
import itertools
from scipy.optimize import minimize, differential_evolution
from sklearn.metrics import mean_squared_error
from astropy.stats import sigma_clip
from scipy.signal import savgol_filter, medfilt, butter, filtfilt
from scipy.ndimage import median_filter, gaussian_filter1d
from scipy.stats import median_abs_deviation
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')
__t0 = time.perf_counter()

In [3]:
class Config:
    DATA_PATH = '/kaggle/input/ariel-data-challenge-2025'
    DATASET = "test"

    SCALE = 0.95
    SIGMA = 0.0009
    
    CUT_INF = 39
    CUT_SUP = 321
    
    SENSOR_CONFIG = {
        "AIRS-CH0": {
            "raw_shape": [11250, 32, 356],
            "calibrated_shape": [1, 32, CUT_SUP - CUT_INF],
            "linear_corr_shape": (6, 32, 356),
            "dt_pattern": (0.1, 4.5), 
            "binning": 30
        },
        "FGS1": {
            "raw_shape": [135000, 32, 32],
            "calibrated_shape": [1, 32, 32],
            "linear_corr_shape": (6, 32, 32),
            "dt_pattern": (0.1, 0.1),
            "binning": 30 * 12
        }
    }
    
    MODEL_PHASE_DETECTION_SLICE = slice(30, 140)
    MODEL_OPTIMIZATION_DELTA = 7
    MODEL_POLYNOMIAL_DEGREE = 3
    
    # Enhanced parameters
    OUTLIER_SIGMA = 3.5
    SMOOTH_WINDOW = 15
    MEDIAN_FILTER_SIZE = 5
    GAUSSIAN_SIGMA = 1.5
    
    N_JOBS = 3

In [4]:
class EnhancedSignalProcessor:
    def __init__(self, config):
        self.cfg = config
        self.adc_info = pd.read_csv(f"{self.cfg.DATA_PATH}/adc_info.csv")
        self.planet_ids = pd.read_csv(f'{self.cfg.DATA_PATH}/{self.cfg.DATASET}_star_info.csv', index_col='planet_id').index.astype(int)

    def _robust_outlier_removal(self, signal, sigma=3.5):
        """Enhanced outlier removal using multiple methods"""
        # Method 1: Sigma clipping
        mask1 = sigma_clip(signal, sigma=sigma, maxiters=3).mask
        
        # Method 2: MAD-based outlier detection
        median = np.median(signal, axis=0)
        mad = median_abs_deviation(signal, axis=0)
        mad[mad == 0] = np.median(mad[mad > 0]) if np.any(mad > 0) else 1.0
        
        z_scores = np.abs(signal - median) / (1.4826 * mad)
        mask2 = z_scores > sigma
        
        # Combine masks
        combined_mask = mask1 | mask2
        return combined_mask

    def _advanced_detrending(self, signal, method='polynomial'):
        """Advanced detrending with multiple methods"""
        if method == 'polynomial':
            # Robust polynomial fitting
            x = np.arange(len(signal))
            coeffs = np.polyfit(x, signal, deg=2)
            trend = np.polyval(coeffs, x)
            
        elif method == 'savgol':
            # Savitzky-Golay with adaptive window
            window_length = min(51, len(signal) // 4)
            if window_length % 2 == 0:
                window_length += 1
            trend = savgol_filter(signal, window_length, 3)
            
        elif method == 'gaussian':
            # Gaussian smoothing
            sigma = len(signal) / 20
            trend = gaussian_filter1d(signal, sigma=sigma)
            
        return signal - trend

    def _apply_linear_corr(self, linear_corr, signal):
        """Enhanced linear correction with bounds checking"""
        coeffs = np.flip(linear_corr, axis=0)
        x = signal.astype(np.float64, copy=False)
        
        # Bounds checking to prevent extreme corrections
        x = np.clip(x, -1e6, 1e6)
        
        out = np.empty_like(x, dtype=np.float64)
        out[...] = coeffs[0]
        
        for k in range(1, coeffs.shape[0]):
            np.multiply(out, x, out=out)
            out += coeffs[k]

        # Additional bounds checking on output
        out = np.clip(out, -1e6, 1e6)
        return out.astype(signal.dtype, copy=False)

    def _calibrate_single_signal(self, planet_id, sensor):
        """Enhanced calibration with better error handling"""
        try:
            sensor_cfg = self.cfg.SENSOR_CONFIG[sensor]

            signal = pd.read_parquet(f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_signal_0.parquet").to_numpy()
            dark = pd.read_parquet(f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_calibration_0/dark.parquet").to_numpy()
            dead = pd.read_parquet(f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_calibration_0/dead.parquet").to_numpy()
            flat = pd.read_parquet(f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_calibration_0/flat.parquet").to_numpy()
            linear_corr = pd.read_parquet(f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_calibration_0/linear_corr.parquet").values.astype(np.float64).reshape(sensor_cfg["linear_corr_shape"])

            signal = signal.reshape(sensor_cfg["raw_shape"])
            gain = self.adc_info[f"{sensor}_adc_gain"].iloc[0]
            offset = self.adc_info[f"{sensor}_adc_offset"].iloc[0]
            
            # Enhanced gain/offset correction with bounds
            gain = max(gain, 1e-10)  # Prevent division by zero
            signal = np.clip(signal / gain + offset, -1e6, 1e6)

            # Enhanced hot pixel detection
            hot = self._robust_outlier_removal(dark, sigma=self.cfg.OUTLIER_SIGMA)

            if sensor == "AIRS-CH0":
                signal = signal[:, :, self.cfg.CUT_INF : self.cfg.CUT_SUP]
                linear_corr = linear_corr[:, :, self.cfg.CUT_INF : self.cfg.CUT_SUP]
                dark = dark[:, self.cfg.CUT_INF : self.cfg.CUT_SUP]
                dead = dead[:, self.cfg.CUT_INF : self.cfg.CUT_SUP]
                flat = flat[:, self.cfg.CUT_INF : self.cfg.CUT_SUP]
                hot = hot[:, self.cfg.CUT_INF : self.cfg.CUT_SUP]

            if sensor == "FGS1":
                y0, y1, x0, x1 = 10, 22, 10, 22
                signal = signal[:, y0:y1, x0:x1]
                dark = dark[y0:y1, x0:x1]
                dead = dead[y0:y1, x0:x1]
                flat = flat[y0:y1, x0:x1]
                linear_corr = linear_corr[:, y0:y1, x0:x1]
                hot = hot[y0:y1, x0:x1]

            # Ensure non-negative values
            np.maximum(signal, 0, out=signal)

            # Apply linear correction
            if sensor == "FGS1":
                signal = self._apply_linear_corr(linear_corr, signal)
            elif sensor == "AIRS-CH0":
                sl = (slice(None), slice(10, 22), slice(None))
                signal[sl] = self._apply_linear_corr(linear_corr[:, 10:22, :], signal[sl])
            else:
                signal = self._apply_linear_corr(linear_corr, signal)

            # Enhanced dark subtraction with pattern correction
            base_dt, increment = sensor_cfg["dt_pattern"]
            even_scale = base_dt
            odd_scale = base_dt + increment

            signal[::2] -= dark * even_scale
            signal[1::2] -= dark * odd_scale
            
            # Remove hot pixels by interpolation
            if np.any(hot):
                signal[:, hot] = np.nan
            
            return signal
            
        except Exception as e:
            print(f"Error processing planet {planet_id}, sensor {sensor}: {e}")
            # Return dummy data to prevent pipeline failure
            dummy_shape = self.cfg.SENSOR_CONFIG[sensor]["raw_shape"]
            return np.zeros(dummy_shape)

    def _advanced_preprocessing(self, calibrated_signal, sensor):
        """Enhanced preprocessing with advanced filtering"""
        sensor_cfg = self.cfg.SENSOR_CONFIG[sensor]
        binning = sensor_cfg["binning"]

        if sensor == "AIRS-CH0":
            signal_roi = calibrated_signal[:, 10:22, :]
        elif sensor == "FGS1":
            signal_roi = calibrated_signal[:, :, :]
            signal_roi = signal_roi.reshape(signal_roi.shape[0], -1)
        
        # Enhanced mean calculation with outlier handling
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            mean_signal = np.nanmean(signal_roi, axis=1)

        # CDS (Correlated Double Sampling) with enhanced noise reduction
        cds_signal = mean_signal[1::2] - mean_signal[0::2]
        
        # Apply median filter to remove cosmic rays
        if self.cfg.MEDIAN_FILTER_SIZE > 1:
            cds_signal = medfilt(cds_signal, kernel_size=self.cfg.MEDIAN_FILTER_SIZE)

        # Binning with overlap for smoother transitions
        n_bins = cds_signal.shape[0] // binning
        if n_bins == 0:
            return np.array([[0.0]])
            
        binned = np.array([
            cds_signal[j*binning : (j+1)*binning].mean(axis=0) 
            for j in range(n_bins)
        ])

        # Enhanced clipping for AIRS-CH0
        if sensor == "AIRS-CH0":
            # More robust percentile-based clipping
            q_lo = np.nanpercentile(binned, 2.5, axis=1, keepdims=True)
            q_hi = np.nanpercentile(binned, 97.5, axis=1, keepdims=True)
            binned = np.clip(binned, q_lo, q_hi)
            
            # Apply Gaussian smoothing to reduce high-frequency noise
            for i in range(binned.shape[1]):
                binned[:, i] = gaussian_filter1d(binned[:, i], sigma=self.cfg.GAUSSIAN_SIGMA)

        if sensor == "FGS1":
            binned = binned.reshape((binned.shape[0], 1))
            # Apply light smoothing to FGS data
            binned[:, 0] = gaussian_filter1d(binned[:, 0], sigma=0.5)

        # Enhanced weighting scheme for AIRS-CH0
        if sensor == "AIRS-CH0":
            # Robust variance estimation
            var = np.nanvar(binned, axis=0, ddof=1)
            
            # Use MAD for robust variance when normal variance fails
            mad_var = (median_abs_deviation(binned, axis=0, nan_policy='omit') * 1.4826) ** 2
            var = np.where(~np.isfinite(var) | (var <= 0), mad_var, var)
            
            med = np.nanmedian(var)
            safe_var = np.where(~np.isfinite(var) | (var <= 0), 
                               med if (np.isfinite(med) and med > 0) else 1.0, var)
            
            w = 1.0 / safe_var

            # More conservative clipping
            lo, hi = np.nanpercentile(w, 10.0), np.nanpercentile(w, 90.0)
            if np.isfinite(lo) and np.isfinite(hi) and lo < hi:
                w = np.clip(w, lo, hi)

            M = binned.shape[1]
            s = np.nansum(w)
            if np.isfinite(s) and s > 0:
                w = w * (M / s)
            else:
                w = np.ones_like(w)

            binned *= w[None, :]
            
        return binned

    def _process_planet_sensor(self, args):
        """Enhanced processing with error handling"""
        planet_id, sensor = args['planet_id'], args['sensor']
        try:
            calibrated = self._calibrate_single_signal(planet_id, sensor)
            preprocessed = self._advanced_preprocessing(calibrated, sensor)
            return preprocessed
        except Exception as e:
            print(f"Error in processing planet {planet_id}, sensor {sensor}: {e}")
            # Return dummy data
            if sensor == "AIRS-CH0":
                return np.zeros((1, self.cfg.CUT_SUP - self.cfg.CUT_INF))
            else:
                return np.zeros((1, 1))

    def process_all_data(self):
        """Enhanced data processing with better error handling"""
        try:
            args_fgs1 = [dict(planet_id=planet_id, sensor="FGS1") for planet_id in self.planet_ids]
            preprocessed_fgs1 = pqdm(args_fgs1, self._process_planet_sensor, n_jobs=self.cfg.N_JOBS)

            args_airs_ch0 = [dict(planet_id=planet_id, sensor="AIRS-CH0") for planet_id in self.planet_ids]
            preprocessed_airs_ch0 = pqdm(args_airs_ch0, self._process_planet_sensor, n_jobs=self.cfg.N_JOBS)

            preprocessed_signal = np.concatenate(
                [np.stack(preprocessed_fgs1), np.stack(preprocessed_airs_ch0)], axis=2
            )
            return preprocessed_signal
        except Exception as e:
            print(f"Error in process_all_data: {e}")
            # Return dummy data to prevent complete failure
            n_planets = len(self.planet_ids)
            dummy_shape = (n_planets, 1, 1 + self.cfg.CUT_SUP - self.cfg.CUT_INF)
            return np.zeros(dummy_shape)

In [5]:
class EnhancedTransitModel:
    def __init__(self, config):
        self.cfg = config

    def _multi_method_phase_detection(self, signal):
        """Enhanced phase detection using multiple methods"""
        search_slice = self.cfg.MODEL_PHASE_DETECTION_SLICE
        
        # Method 1: Original gradient-based
        min_index = np.argmin(signal[search_slice]) + search_slice.start
        signal1 = signal[:min_index]
        signal2 = signal[min_index:]

        if len(signal1) < 3 or len(signal2) < 3:
            return max(10, len(signal) // 4), min(len(signal) - 10, 3 * len(signal) // 4)

        grad1 = np.gradient(signal1)
        grad2 = np.gradient(signal2)
        
        # Normalize gradients safely
        if grad1.max() > 0:
            grad1 /= grad1.max()
        if grad2.max() > 0:
            grad2 /= grad2.max()

        phase1_grad = np.argmin(grad1)
        phase2_grad = np.argmax(grad2) + min_index

        # Method 2: Second derivative based
        second_deriv = np.gradient(np.gradient(signal))
        smoothed_second = savgol_filter(second_deriv, min(21, len(signal)//3), 2)
        
        # Find zero crossings in second derivative
        zero_crossings = np.where(np.diff(np.sign(smoothed_second)))[0]
        if len(zero_crossings) >= 2:
            phase1_deriv = zero_crossings[0]
            phase2_deriv = zero_crossings[-1]
        else:
            phase1_deriv, phase2_deriv = phase1_grad, phase2_grad

        # Method 3: Template matching (simplified box model)
        box_width = max(10, len(signal) // 6)
        best_fit = float('inf')
        best_phase1, best_phase2 = phase1_grad, phase2_grad
        
        for start in range(max(5, len(signal)//4), min(len(signal)-box_width-5, 3*len(signal)//4)):
            template = np.ones_like(signal)
            template[start:start+box_width] *= 0.95  # 5% transit depth
            
            residual = np.mean((signal - template)**2)
            if residual < best_fit:
                best_fit = residual
                best_phase1, best_phase2 = start, start + box_width

        # Combine methods with weighted average
        phases = np.array([[phase1_grad, phase2_grad], 
                          [phase1_deriv, phase2_deriv],
                          [best_phase1, best_phase2]])
        
        # Weight by method reliability (gradient method gets highest weight)
        weights = np.array([0.5, 0.3, 0.2])
        
        final_phase1 = int(np.average(phases[:, 0], weights=weights))
        final_phase2 = int(np.average(phases[:, 1], weights=weights))
        
        # Ensure reasonable bounds
        final_phase1 = max(self.cfg.MODEL_OPTIMIZATION_DELTA, final_phase1)
        final_phase2 = min(len(signal) - self.cfg.MODEL_OPTIMIZATION_DELTA - 1, final_phase2)
        
        return final_phase1, final_phase2

    def _enhanced_objective_function(self, params, signal, phase1, phase2):
        """Enhanced objective function with multiple components"""
        if len(params) == 1:
            s = params[0]
            baseline_slope = 0.0
        else:
            s, baseline_slope = params
        
        delta = self.cfg.MODEL_OPTIMIZATION_DELTA
        power = self.cfg.MODEL_POLYNOMIAL_DEGREE

        # Adaptive delta for edge cases
        if phase1 - delta <= 0 or phase2 + delta >= len(signal) or phase2 - delta - (phase1 + delta) < 5:
            delta = max(1, min(delta, (phase2 - phase1) // 4))

        # Bounds checking
        if abs(s) > 0.2:  # Reasonable transit depth limit
            return 1e6
        
        if abs(baseline_slope) > 0.01:  # Reasonable baseline slope limit
            return 1e6

        try:
            # Create the corrected signal with baseline trend
            x_all = np.arange(len(signal))
            baseline_correction = baseline_slope * (x_all - len(signal)/2)
            
            y = signal.copy() + baseline_correction
            y[phase1 + delta : phase2 - delta] *= (1 + s)
            
            # Exclude transit region from polynomial fit
            mask = np.ones(len(y), dtype=bool)
            mask[phase1 - delta : phase2 + delta] = False
            
            if np.sum(mask) < power + 1:
                return 1e6
            
            x_fit = x_all[mask]
            y_fit = y[mask]
            
            # Robust polynomial fitting
            coeffs = np.polyfit(x_fit, y_fit, deg=power)
            poly = np.poly1d(coeffs)
            
            # Calculate residuals
            y_model = poly(x_all)
            residuals = y - y_model
            
            # Multi-component objective
            mse = np.mean(residuals**2)
            
            # Regularization terms
            depth_penalty = 0.01 * s**2  # Prefer smaller depths
            slope_penalty = 0.1 * baseline_slope**2  # Prefer flat baselines
            smoothness_penalty = 0.001 * np.mean(np.diff(residuals)**2)  # Prefer smooth residuals
            
            total_objective = mse + depth_penalty + slope_penalty + smoothness_penalty
            
            return total_objective
            
        except Exception as e:
            return 1e6

    def _bayesian_optimization(self, signal, phase1, phase2):
        """Enhanced optimization with multiple methods"""
        
        # Method 1: Nelder-Mead (original)
        try:
            result1 = minimize(
                fun=self._enhanced_objective_function,
                x0=[0.0001],
                args=(signal, phase1, phase2),
                method="Nelder-Mead",
                options={'maxiter': 100}
            )
            score1 = result1.fun if result1.success else 1e6
        except:
            score1 = 1e6
            result1 = type('obj', (object,), {'x': [0.0001], 'fun': 1e6})()

        # Method 2: L-BFGS-B with bounds
        try:
            result2 = minimize(
                fun=self._enhanced_objective_function,
                x0=[0.0001, 0.0],
                args=(signal, phase1, phase2),
                method="L-BFGS-B",
                bounds=[(-0.15, 0.05), (-0.005, 0.005)],
                options={'maxiter': 50}
            )
            score2 = result2.fun if result2.success else 1e6
        except:
            score2 = 1e6
            result2 = type('obj', (object,), {'x': [0.0001, 0.0], 'fun': 1e6})()

        # Method 3: Differential Evolution (global optimizer)
        try:
            result3 = differential_evolution(
                func=self._enhanced_objective_function,
                bounds=[(-0.15, 0.05)],
                args=(signal, phase1, phase2),
                seed=42,
                maxiter=30,
                popsize=5
            )
            score3 = result3.fun if result3.success else 1e6
        except:
            score3 = 1e6
            result3 = type('obj', (object,), {'x': [0.0001], 'fun': 1e6})()

        # Choose best result
        scores = [score1, score2, score3]
        results = [result1, result2, result3]
        
        best_idx = np.argmin(scores)
        best_result = results[best_idx]
        
        return best_result.x[0] if len(best_result.x) >= 1 else 0.0001

    def predict(self, single_preprocessed_signal):
        """Enhanced prediction with robust processing"""
        try:
            # Multi-wavelength combination for better SNR
            if single_preprocessed_signal.shape[1] > 1:
                # Weighted combination of wavelengths
                weights = np.ones(single_preprocessed_signal.shape[1] - 1)
                signal_1d = np.average(single_preprocessed_signal[:, 1:], axis=1, weights=weights)
            else:
                signal_1d = single_preprocessed_signal[:, 0]
            
            # Enhanced filtering
            if len(signal_1d) > 10:
                # Apply multiple filters and combine
                savgol_filtered = savgol_filter(signal_1d, min(21, len(signal_1d)//2), 2)
                gaussian_filtered = gaussian_filter1d(signal_1d, sigma=1.0)
                
                # Weighted combination of filters
                signal_1d = 0.7 * savgol_filtered + 0.3 * gaussian_filtered
            
            # Enhanced phase detection
            phase1, phase2 = self._multi_method_phase_detection(signal_1d)
            
            # Enhanced optimization
            transit_depth = self._bayesian_optimization(signal_1d, phase1, phase2)
            
            return transit_depth
            
        except Exception as e:
            print(f"Error in prediction: {e}")
            return 0.0001  # Default small transit depth

    def predict_all(self, preprocessed_signals):
        """Enhanced batch prediction with progress tracking"""
        predictions = []
        failed_predictions = 0
        
        for i, preprocessed_signal in enumerate(tqdm(preprocessed_signals, desc="Predicting transits")):
            try:
                prediction = self.predict(preprocessed_signal)
                predictions.append(prediction)
            except Exception as e:
                print(f"Failed prediction for signal {i}: {e}")
                predictions.append(0.0001)
                failed_predictions += 1
        
        if failed_predictions > 0:
            print(f"Warning: {failed_predictions} predictions failed and were set to default values")
        
        return np.array(predictions) * self.cfg.SCALE

In [6]:
def estimate_sigma_fgs_enhanced(preprocessed_data, cfg):
    """Enhanced FGS sigma estimation with robust statistics"""
    sig_rel = []
    delta = cfg.MODEL_OPTIMIZATION_DELTA
    eps = 1e-12
    
    for single in preprocessed_data:
        try:
            air_white = savgol_filter(single[:, 1:].mean(axis=1), 20, 2)
            phase1, phase2 = _phase_detector_signal_enhanced(air_white, cfg)
            phase1 = max(delta, phase1)
            phase2 = min(len(air_white) - delta - 1, phase2)

            fgs = single[:, 0]
            oot = (fgs[: phase1 - delta] if phase1 - delta > 0 else np.empty(0, fgs.dtype))
            if phase2 + delta < fgs.size:
                oot = np.concatenate([oot, fgs[phase2 + delta :]])
            inn = fgs[phase1 + delta : max(phase1 + delta, phase2 - delta)]

            if oot.size == 0 or inn.size == 0:
                sig_rel.append(np.nan)
                continue

            # Use MAD for robust variance estimation
            mad_oot = median_abs_deviation(oot, nan_policy='omit')
            mad_in = median_abs_deviation(inn, nan_policy='omit')
            
            var_oot = (1.4826 * mad_oot) ** 2
            var_in = (1.4826 * mad_in) ** 2
            
            n_oot, n_in = len(oot), len(inn)
            oot_mean = float(np.nanmean(oot)) if np.isfinite(np.nanmean(oot)) else float(np.nanmean(fgs))
            
            sigma_rel = np.sqrt(var_oot / max(n_oot,1) + var_in / max(n_in,1)) / max(oot_mean, eps)
            sig_rel.append(sigma_rel)
        except:
            sig_rel.append(np.nan)

    s = np.asarray(sig_rel, dtype=float)
    mask = np.isfinite(s) & (s > 0)
    med = float(np.nanmedian(s[mask])) if mask.any() else 1.0

    k = np.ones_like(s)
    if med > 0 and np.isfinite(med):
        k[mask] = np.sqrt(s[mask] / med)
    k = np.clip(k, 0.7, 1.4)  # Slightly wider range

    return k * cfg.SIGMA

def _phase_detector_signal_enhanced(signal, cfg):
    """Enhanced phase detection for sigma estimation"""
    try:
        sl = cfg.MODEL_PHASE_DETECTION_SLICE
        min_idx = int(np.argmin(signal[sl])) + sl.start
        s1 = signal[:min_idx]
        s2 = signal[min_idx:]
        
        if s1.size < 3 or s2.size < 3:
            return 0, len(signal) - 1
            
        g1 = np.gradient(s1)
        g2 = np.gradient(s2)
        
        g1_max = np.max(g1) if np.size(g1) else 0.0
        g2_max = np.max(g2) if np.size(g2) else 0.0
        
        if g1_max != 0:
            g1 /= g1_max
        if g2_max != 0:
            g2 /= g2_max
            
        phase1 = int(np.argmin(g1))
        phase2 = int(np.argmax(g2)) + min_idx
        
        return phase1, phase2
    except:
        return 0, len(signal) - 1

def estimate_sigma_air_enhanced(preprocessed_data, cfg):
    """Enhanced AIRS sigma estimation with robust statistics"""
    sig_rel = []
    delta = cfg.MODEL_OPTIMIZATION_DELTA
    eps = 1e-12

    for single in preprocessed_data:
        try:
            white = np.nanmean(single[:, 1:], axis=1)
            white_s = savgol_filter(white, 20, 2)

            phase1, phase2 = _phase_detector_signal_enhanced(white_s, cfg)
            phase1 = max(delta, phase1)
            phase2 = min(len(white) - delta - 1, phase2)

            oot_left = white[: phase1 - delta] if phase1 - delta > 0 else np.empty(0, white.dtype)
            oot_right = white[phase2 + delta :] if (phase2 + delta) < white.size else np.empty(0, white.dtype)
            oot = np.concatenate([oot_left, oot_right]) if (oot_left.size + oot_right.size) else oot_left
            inn = white[phase1 + delta : max(phase1 + delta, phase2 - delta)]

            if oot.size == 0 or inn.size == 0:
                sig_rel.append(np.nan)
                continue

            # Use MAD for robust variance estimation
            mad_oot = median_abs_deviation(oot, nan_policy='omit')
            mad_in = median_abs_deviation(inn, nan_policy='omit')
            
            var_oot = (1.4826 * mad_oot) ** 2
            var_in = (1.4826 * mad_in) ** 2
            
            n_oot, n_in = len(oot), len(inn)
            oot_mean = float(np.nanmean(oot)) if np.isfinite(np.nanmean(oot)) else float(np.nanmean(white))

            sigma_rel = np.sqrt(var_oot / max(n_oot,1) + var_in / max(n_in,1)) / max(oot_mean, eps)
            sig_rel.append(sigma_rel)
        except:
            sig_rel.append(np.nan)

    s = np.asarray(sig_rel, dtype=float)
    mask = np.isfinite(s) & (s > 0)
    med = float(np.nanmedian(s[mask])) if mask.any() else 1.0

    k = np.ones_like(s)
    if med > 0 and np.isfinite(med):
        k[mask] = np.sqrt(s[mask] / med)
    k = np.clip(k, 0.8, 1.3)  # Slightly wider range

    return k * cfg.SIGMA

In [7]:
class EnhancedSubmissionGenerator:
    def __init__(self, config):
        self.cfg = config
        self.sample_submission = pd.read_csv("/kaggle/input/ariel-data-challenge-2025/sample_submission.csv", index_col="planet_id")

    def create(self, predictions, sigma_fgs=None, sigma_air=None):
        """Enhanced submission creation with quality checks"""
        planet_ids = self.sample_submission.index
        n_mu = self.sample_submission.shape[1] // 2

        preds = np.asarray(predictions, dtype=float).reshape(-1)
        
        # Quality checks on predictions
        preds = np.where(~np.isfinite(preds), 0.0001, preds)  # Replace NaN/inf
        preds = np.clip(preds, -0.1, 0.1)  # Reasonable bounds for transit depths
        
        mu = np.tile(preds.reshape(-1, 1), (1, n_mu))
        mu = np.clip(mu, 0, None)  # Ensure non-negative

        sigmas = np.full_like(mu, self.cfg.SIGMA, dtype=float)
        
        if sigma_fgs is not None:
            sigma_fgs = np.asarray(sigma_fgs, dtype=float).reshape(-1)
            sigma_fgs = np.where(~np.isfinite(sigma_fgs), self.cfg.SIGMA, sigma_fgs)
            sigmas[:, 0] = np.clip(sigma_fgs, 1e-6, 0.1)
            
        if sigma_air is not None:
            sigma_air = np.asarray(sigma_air, dtype=float).reshape(-1, 1)
            sigma_air = np.where(~np.isfinite(sigma_air), self.cfg.SIGMA, sigma_air)
            sigmas[:, 1:] = np.clip(sigma_air, 1e-6, 0.1)

        submission_df = pd.DataFrame(
            np.concatenate([mu, sigmas], axis=1),
            columns=self.sample_submission.columns,
            index=planet_ids
        )
        
        # Final quality check
        submission_df = submission_df.fillna(self.cfg.SIGMA)
        submission_df = submission_df.clip(lower=0)
        
        submission_df.to_csv("submission.csv")
        
        # Print summary statistics
        print(f"Submission Statistics:")
        print(f"Mean transit depth: {np.mean(preds):.6f}")
        print(f"Std transit depth: {np.std(preds):.6f}")
        print(f"Min/Max transit depth: {np.min(preds):.6f} / {np.max(preds):.6f}")
        print(f"Number of significant detections (>3σ): {np.sum(np.abs(preds) > 3*self.cfg.SIGMA)}")
        
        return submission_df

# Quality assessment function
def assess_data_quality(preprocessed_data, config):
    """Assess the quality of processed data"""
    quality_scores = []
    
    for i, single in enumerate(preprocessed_data):
        try:
            # Check for NaN values
            nan_fraction = np.mean(~np.isfinite(single))
            
            # Check signal-to-noise ratio
            if single.shape[1] > 1:
                signal = np.nanmean(single[:, 1:], axis=1)
            else:
                signal = single[:, 0]
                
            snr = np.nanstd(signal) / np.nanmean(np.abs(np.diff(signal)))
            
            # Check for outliers
            z_scores = np.abs((signal - np.nanmean(signal)) / np.nanstd(signal))
            outlier_fraction = np.mean(z_scores > 3)
            
            # Composite quality score (higher is better)
            quality = (1 - nan_fraction) * min(snr/10, 1.0) * (1 - outlier_fraction)
            quality_scores.append(quality)
            
        except:
            quality_scores.append(0.0)
    
    quality_scores = np.array(quality_scores)
    print(f"Data Quality Assessment:")
    print(f"Mean quality score: {np.mean(quality_scores):.3f}")
    print(f"High quality signals (>0.7): {np.sum(quality_scores > 0.7)}/{len(quality_scores)}")
    print(f"Low quality signals (<0.3): {np.sum(quality_scores < 0.3)}/{len(quality_scores)}")
    
    return quality_scores

In [8]:
try:
    print("Starting enhanced ARIEL data processing pipeline...")
    
    config = Config()
    
    print("Initializing enhanced signal processor...")
    signal_processor = EnhancedSignalProcessor(config)
    
    print("Processing all signals with advanced calibration...")
    preprocessed_data = signal_processor.process_all_data()
    
    print("Assessing data quality...")
    quality_scores = assess_data_quality(preprocessed_data, config)
    
    print("Initializing enhanced transit model...")
    model = EnhancedTransitModel(config)
    
    print("Making predictions with advanced transit modeling...")
    predictions = model.predict_all(preprocessed_data)
    
    print("Estimating uncertainties with robust statistics...")
    sigma_fgs_vec = estimate_sigma_fgs_enhanced(preprocessed_data, config)
    sigma_air_vec = estimate_sigma_air_enhanced(preprocessed_data, config)
    
    print("Generating enhanced submission...")
    submission_generator = EnhancedSubmissionGenerator(config)
    submission = submission_generator.create(predictions, sigma_fgs=sigma_fgs_vec, sigma_air=sigma_air_vec)
    
    __t1 = time.perf_counter()
    elapsed = __t1 - __t0
    print(f"\n[TIMING] Enhanced pipeline completed in: {elapsed:.2f} s ({elapsed/60:.2f} min)")
    print("Enhanced submission saved as 'submission.csv'")
    
except Exception as e:
    print(f"Critical error in enhanced pipeline: {e}")
    import traceback
    traceback.print_exc()
    
    # Fallback: create basic submission
    try:
        config = Config()
        sample_submission = pd.read_csv("/kaggle/input/ariel-data-challenge-2025/sample_submission.csv", index_col="planet_id")
        fallback_submission = sample_submission.copy()
        fallback_submission.iloc[:, :] = config.SIGMA
        fallback_submission.to_csv("submission.csv")
        print("Fallback submission created with default values.")
    except Exception as fallback_error:
        print(f"Even fallback failed: {fallback_error}")

print("Enhanced ARIEL pipeline execution completed.")
            

Starting enhanced ARIEL data processing pipeline...
Initializing enhanced signal processor...
Processing all signals with advanced calibration...


QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

Assessing data quality...
Data Quality Assessment:
Mean quality score: 1.000
High quality signals (>0.7): 1/1
Low quality signals (<0.3): 0/1
Initializing enhanced transit model...
Making predictions with advanced transit modeling...


Predicting transits: 100%|██████████| 1/1 [00:00<00:00,  3.63it/s]

Estimating uncertainties with robust statistics...
Generating enhanced submission...
Submission Statistics:
Mean transit depth: 0.015220
Std transit depth: 0.000000
Min/Max transit depth: 0.015220 / 0.015220
Number of significant detections (>3σ): 1

[TIMING] Enhanced pipeline completed in: 11.43 s (0.19 min)
Enhanced submission saved as 'submission.csv'
Enhanced ARIEL pipeline execution completed.
